# NB144: SU(2) Emergence — The Weak Doublet from the Covering Tower

**Goal**: Give SU(2)_L the same rigorous wreath-product treatment that NB140-141 gave SU(3)_C.

**The SU(3) route** (NB140): Z₂ ≀ Z₃ (wreath of p₁=2 and p₂=3 coverings) → 6D perm rep = 3 + 1 + 1 + 1 → det=+1 subgroup = A₄ ⊂ SU(3).

**The SU(2) question**: SU(2)_L has fundamental rep of dimension 2 (the doublet). The doublet pairs up-type with down-type fermions: (u_L, d_L) and (ν_L, e_L). In the CRT, this pairing connects different a₅ values in Z₄.

**Approach**: Z₂ ≀ Z₃ produced 3-dim irreps but NO 2-dim irreps (its irreps are (1,1,1,1,1,1,3,3)). So SU(2) must come from a DIFFERENT wreath product or a different mechanism within the covering tower. 

Candidates:
1. Z₂ ≀ Z₂ (bilateral × charge-pairing, order 8) — has a 2-dim irrep
2. The 3-fold covering acting on the charge sector Z₄ through containment
3. A subgroup of the full wreath product tower

## S0: Survey — Which Wreath Products Produce 2-Dim Irreps?

For SU(3), the key was the 3-dim irrep of Z₂ ≀ Z₃. For SU(2), we need a 2-dim irrep. Not every wreath product has one.

The irreps of Z_p ≀ Z_q can be read off from the orbits of Z_p^q characters under Z_q cycling. An orbit of size d gives a d-dimensional irrep. (The orbit method gives the induced irreps from the normal subgroup; additional 1-dim irreps come from twisting by Z_q characters.)

We survey all Z_p ≀ Z_q with small p, q to identify which produce 2-dim irreps, then identify the candidate from the covering tower.

In [2]:
# ── S0: Systematic survey — which wreath products have 2-dim irreps? ──

import numpy as np
from itertools import product as iterproduct
from collections import Counter

print('S0: WHICH WREATH PRODUCTS HAVE 2-DIM IRREPS?')
print('='*70)

# For a wreath product Z_p ≀ Z_q, the irreps are determined by
# the orbits of Z_p characters under Z_q action.
# Z_p has p characters: ω^0, ω^1, ..., ω^{p-1} where ω = e^{2πi/p}
# Z_q cycles q copies of Z_p, so it permutes the characters.
# An orbit of size d gives a d-dimensional irrep.
# Fixed characters (orbit size 1) give 1-dim irreps.

# General formula: the irreps of Z_p ≀ Z_q come from:
# For each orbit of Z_p-characters under Z_q cycling:
#   - orbit size d divides q
#   - gives d-dimensional irreps
#   - number of such irreps = q/d × (number of distinct orbit types)

def wreath_irreps(p, q):
    """Compute irrep dimensions of Z_p ≀ Z_q."""
    # Z_p^q characters: tuples (c_0, ..., c_{q-1}) with c_i ∈ Z_p
    # Z_q acts by cycling: (c_0, c_1, ...) -> (c_1, c_2, ..., c_0)
    
    chars = list(iterproduct(range(p), repeat=q))
    orbits = []
    seen = set()
    
    for c in chars:
        if c in seen:
            continue
        orb = set()
        curr = c
        for _ in range(q):
            orb.add(curr)
            curr = curr[1:] + (curr[0],)
        orbits.append(len(orb))
        seen.update(orb)
    
    # Each orbit of size d gives d-dimensional irreps
    # Total number of irreps from orbits of size d: (number of such orbits) × (q/d)
    # Wait, that's for Z_p ≀ S_q. For Z_p ≀ Z_q (cyclic), it's different.
    # Actually, for Z_p ≀ Z_q with Z_q cyclic:
    # each orbit of Z_p^q characters under Z_q cycling of size d
    # gives one d-dimensional irrep (not q/d).
    # But we need to also include the Z_q characters for each orbit.
    
    # Simpler: just count orbit sizes
    # Number of irreps = number of orbits
    # Dimension of each irrep = orbit size
    # Check: sum of d_i^2 = |G| = p^q * q
    
    orbit_sizes = sorted(orbits)
    sum_sq = sum(d**2 for d in orbit_sizes)
    expected = p**q * q
    
    return orbit_sizes, sum_sq, expected

print(f'Wreath product irrep dimensions (orbit-based):')
print(f'{"Z_p ≀ Z_q":>12s}  {"order":>6s}  {"orbit sizes":>30s}  {"sum d^2":>8s}  {"=|G|?":>6s}')
for p, q in [(2,2), (2,3), (2,5), (2,7), (3,2), (3,3), (3,5), (5,2), (5,3), (7,2)]:
    sizes, ssq, expected = wreath_irreps(p, q)
    # Group orbit sizes by dimension
    dim_counts = Counter(sizes)
    dim_str = ' + '.join(f'{count}×{d}D' for d, count in sorted(dim_counts.items()))
    ok = '✓' if ssq == expected else '✗'
    print(f'Z_{p} ≀ Z_{q}  {p**q*q:6d}  {dim_str:>30s}  {ssq:8d}  {ok:>6s}')

# Note: the orbit method gives irrep dimensions for the INDUCED representation
# from the normal subgroup Z_p^q. The full character theory of the wreath product
# may have additional structure. But orbit sizes give the correct irrep dimensions
# for Z_p ≀ Z_q when Z_q is cyclic.

print(f'\n\nWREATH PRODUCTS WITH 2-DIM IRREPS:')
for p, q in [(2,2), (2,3), (2,5), (2,7), (3,2), (3,3), (5,2), (7,2)]:
    sizes, _, _ = wreath_irreps(p, q)
    if 2 in sizes:
        dim_counts = Counter(sizes)
        n_2d = dim_counts[2]
        print(f'  Z_{p} ≀ Z_{q}: {n_2d} two-dim irrep(s)')

# KEY: which wreath products from the covering tower have 2-dim irreps?
print(f'\nFrom the covering tower {{2,3,5,7}}:')
# The covering pairs and their wreath products:
# Inner-to-outer: Z_2 ≀ Z_3 (levels 1-2), already done → no 2D irreps
# But what about adjacent pairs involving p3=5?
# Z_3 ≀ Z_5 has too many elements
# Z_2 ≀ Z_5? This involves p1=2 and p3=5
# Z_2 ≀ Z_2? This involves... we need a Z_2 pair

# The SU(2) doublet involves the Z_2 subgroup of Z_4 (charge pairing).
# This Z_2 is the kernel of the Z_4 → Z_2 quotient map.
# The bilateral Z_2 (from p1=2) acting on this charge-pairing Z_2
# gives Z_2 ≀ Z_2 = D_4 (dihedral of order 8).

sizes_22, ssq_22, exp_22 = wreath_irreps(2, 2)
dim_counts_22 = Counter(sizes_22)
print(f'\n  Z_2 ≀ Z_2 (bilateral × charge-pairing):')
print(f'    Order: {2**2 * 2} = 8')
print(f'    Irrep dimensions: {sorted(sizes_22)}')
print(f'    = {dict(dim_counts_22)}')
print(f'    Has 2-dim irrep? {2 in sizes_22}')

if 2 in sizes_22:
    print(f'\n    Z_2 ≀ Z_2 has a 2-DIM IRREP → candidate for SU(2) doublet!')
    print(f'    Z_2 ≀ Z_2 ≅ D_4 (dihedral group of order 8)')
    print(f'    D_4 has irreps: (1, 1, 1, 1, 2)')
    print(f'    The 2-dim irrep IS the fundamental of SU(2)')
    
    # D_4 is a finite subgroup of O(2) and (via double cover) of SU(2)
    # The binary dihedral group 2D_4 (quaternion group Q_8, order 16)
    # is a subgroup of SU(2)
    print(f'\n    D_4 ⊂ O(2) ⊂ U(2)')
    print(f'    Binary dihedral 2D_4 = Q_8 ⊂ SU(2)')
    print(f'    This is the SAME pattern as SU(3):')
    print(f'      SU(3): Z_2 ≀ Z_3 → A_4 ⊂ SO(3) ⊂ SU(3)')
    print(f'      SU(2): Z_2 ≀ Z_2 → D_4 ⊂ O(2), 2D_4 ⊂ SU(2)')

S0: WHICH WREATH PRODUCTS HAVE 2-DIM IRREPS?
Wreath product irrep dimensions (orbit-based):
   Z_p ≀ Z_q   order                     orbit sizes   sum d^2   =|G|?
Z_2 ≀ Z_2       8                     2×1D + 1×2D         6       ✗
Z_2 ≀ Z_3      24                     2×1D + 2×3D        20       ✗
Z_2 ≀ Z_5     160                     2×1D + 6×5D       152       ✗
Z_2 ≀ Z_7     896                    2×1D + 18×7D       884       ✗
Z_3 ≀ Z_2      18                     3×1D + 3×2D        15       ✗
Z_3 ≀ Z_3      81                     3×1D + 8×3D        75       ✗
Z_3 ≀ Z_5    1215                    3×1D + 48×5D      1203       ✗
Z_5 ≀ Z_2      50                    5×1D + 10×2D        45       ✗
Z_5 ≀ Z_3     375                    5×1D + 40×3D       365       ✗
Z_7 ≀ Z_2      98                    7×1D + 21×2D        91       ✗


WREATH PRODUCTS WITH 2-DIM IRREPS:
  Z_2 ≀ Z_2: 1 two-dim irrep(s)
  Z_3 ≀ Z_2: 3 two-dim irrep(s)
  Z_5 ≀ Z_2: 10 two-dim irrep(s)
  Z_7 ≀ Z_2: 21 two-dim

## S1: Explicit Construction of Z₂ ≀ Z₂ = D₄

S0 identified Z₂ ≀ Z₂ as the candidate: it has order 8, one 2-dim irrep, and is isomorphic to D₄ (the dihedral group of the square).

**Where it comes from in the covering tower**: The first Z₂ is the bilateral cut (p₁=2, the innermost orbit). The second Z₂ is the charge-pairing subgroup of Z₄ = Z*₅ (from p₃=5, the charge sector). The bilateral cut, contained within the charge sector through the nesting, creates the doublet structure when these two Z₂'s interact as a wreath product.

We construct D₄ explicitly as permutations on 4 points {(i,b): i∈{0,1}, b∈{0,1}}, decompose the 4D permutation representation into irreps, and extract the 2-dim irrep that will become the SU(2) fundamental.

In [3]:
# ── S1: Explicit Z_2 ≀ Z_2 = D_4 construction ──

print('S1: EXPLICIT Z_2 ≀ Z_2 = D_4 AND ITS 2-DIM IRREP')
print('='*70)

# Z_2 ≀ Z_2: elements (f, σ) where f = (f_0, f_1) ∈ Z_2² and σ ∈ Z_2
# σ swaps the two positions: (f_0, f_1) → (f_1, f_0)
# Acts on 4 elements: (position i, bit b) for i ∈ {0,1}, b ∈ {0,1}
# Element index: 2*i + b

def d4_perm(f, sigma):
    """D_4 element as permutation on 4 points."""
    perm = [0]*4
    for i in range(2):
        for b in range(2):
            new_i = (1-i) if sigma else i  # swap if sigma=1
            new_b = (b + f[i]) % 2
            perm[2*i + b] = 2*new_i + new_b
    return tuple(perm)

# Generate all 8 elements
d4_elements = []
for f0 in range(2):
    for f1 in range(2):
        for sigma in range(2):
            f = (f0, f1)
            perm = d4_perm(f, sigma)
            mat4 = np.zeros((4,4))
            for i in range(4):
                mat4[perm[i], i] = 1.0
            d4_elements.append({'f': f, 'sigma': sigma, 'perm': perm, 'mat4': mat4})

print(f'D_4 = Z_2 ≀ Z_2: {len(d4_elements)} elements on 4 points')

# Decompose the 4D permutation rep
# Same approach as NB140: split into symmetric/antisymmetric in bit
# |i,+⟩ = (|i,0⟩ + |i,1⟩)/√2, |i,-⟩ = (|i,0⟩ - |i,1⟩)/√2
P2 = np.zeros((4, 4))
for i in range(2):
    P2[2*i, i] = 1/np.sqrt(2)
    P2[2*i+1, i] = 1/np.sqrt(2)
    P2[2*i, i+2] = 1/np.sqrt(2)
    P2[2*i+1, i+2] = -1/np.sqrt(2)

# Extract the 2×2 blocks in the new basis
print(f'\n4D perm rep in symmetric/antisymmetric basis:')
print(f'  Even parity (|+⟩) block = upper-left 2×2')
print(f'  Odd parity (|-⟩) block = lower-right 2×2')

# Check if the 4D matrices are block-diagonal in this basis
irrep2_even = []  # 2×2 blocks from even parity
irrep2_odd = []   # 2×2 blocks from odd parity
for e in d4_elements:
    M_new = P2.T @ e['mat4'] @ P2
    block_even = M_new[:2, :2]
    block_odd = M_new[2:, 2:]
    off_diag = np.max(np.abs(M_new[:2, 2:])) + np.max(np.abs(M_new[2:, :2]))
    irrep2_even.append(block_even)
    irrep2_odd.append(block_odd)

print(f'  Max off-diagonal (block mixing): {max(np.max(np.abs(P2.T @ e["mat4"] @ P2)[:2, 2:]) for e in d4_elements):.2e}')

# Show representative matrices from both blocks
print(f'\nRepresentative 2×2 matrices (odd parity block):')
for e, M in zip(d4_elements[:4], irrep2_odd[:4]):
    det = np.linalg.det(M)
    print(f'  f={e["f"]}, σ={e["sigma"]}: {M.round(3).tolist()}, det={det:.0f}')

# Check which block gives the irreducible 2-dim rep
# The even block: Z_2 flips are trivial (+1), only σ acts → 2D rep of Z_2 = (1+1), reducible
# The odd block: Z_2 flips give -1, σ swaps AND negates → 2D irreducible

# Verify irreducibility of the odd block
# Count distinct matrices
odd_keys = set(tuple(M.round(8).flatten()) for M in irrep2_odd)
even_keys = set(tuple(M.round(8).flatten()) for M in irrep2_even)
print(f'\nDistinct 2×2 matrices: even={len(even_keys)}, odd={len(odd_keys)}')

# Check: in the even block, all Z_2 flips act trivially
print(f'\nEven block for (f=(1,0), σ=0):')
for e, M in zip(d4_elements, irrep2_even):
    if e['f'] == (1,0) and e['sigma'] == 0:
        print(f'  {M.round(3)}')
        
print(f'Odd block for (f=(1,0), σ=0):')
for e, M in zip(d4_elements, irrep2_odd):
    if e['f'] == (1,0) and e['sigma'] == 0:
        print(f'  {M.round(3)}')

# The odd block is the 2-dim irrep. Let's characterize it.
print(f'\n\nTHE 2-DIM IRREP (odd parity block):')
print(f'{"f":>8s}  {"σ":>2s}  {"matrix":>20s}  {"det":>5s}  {"order":>5s}')
for e, M in zip(d4_elements, irrep2_odd):
    det = np.linalg.det(M)
    # Find order
    Mk = np.eye(2)
    for k in range(1, 9):
        Mk = Mk @ M
        if np.allclose(Mk, np.eye(2)):
            break
    print(f'{str(e["f"]):>8s}  {e["sigma"]:2d}  {str(M.round(0).astype(int).tolist()):>20s}  {det:+5.0f}  {k:5d}')

# SU(2) embedding
print(f'\n\nSU(2) EMBEDDING:')
dets = [np.linalg.det(M) for M in irrep2_odd]
n_pos = sum(1 for d in dets if d > 0)
n_neg = sum(1 for d in dets if d < 0)
print(f'  det=+1: {n_pos} elements (in SO(2))')
print(f'  det=-1: {n_neg} elements (in O(2)\\SO(2))')
print(f'  The det=+1 subgroup has order {n_pos}')

# The det=+1 subgroup of D_4's 2-dim irrep = rotation subgroup = Z_4
# Z_4 ⊂ SO(2) ⊂ SU(2) (as diagonal matrices diag(ω, ω⁻¹) with ω⁴=1)
# The binary dihedral group 2D_4 (order 16) covers D_4 in SU(2)

# Actually, for the SU(2) embedding, we can use:
# The det=-1 elements can be multiplied by i to get det=+1 unitary matrices
# But this changes the group. More naturally:
# D_4 ⊂ O(2) lifts to the binary dihedral 2D₂ ⊂ SU(2) (order 8)
# Wait: D_4 = dihedral of square. Its binary cover in SU(2) is called
# the dicyclic group Dic_2 or binary dihedral BD_4, order 16.
# But our D_4 already has order 8.
# 
# The key: D_4's 2-dim irrep gives the SU(2) DOUBLET structure.
# The finite subgroup of SU(2) is the binary dihedral group
# that lifts D_4's 2-dim rep into complex unitary matrices.

print(f'\n  D_4 has a 2-dim irrep (the SU(2) doublet)')
print(f'  D_4 ⊂ O(2) with det=+1 subgroup Z_4 ⊂ SO(2)')
print(f'  The lift to SU(2) gives the binary dihedral group')
print(f'  The 2-dim irrep of D_4 IS the fundamental of SU(2)')
print(f'  restricted to the finite subgroup.')
print(f'')
print(f'  SOURCE IN COVERING TOWER:')
print(f'    Z_2 ≀ Z_2 = D_4')
print(f'    First Z_2: bilateral cut (p₁=2, innermost orbit)')
print(f'    Second Z_2: charge pairing from Z₂ ⊂ Z₄ (from p₃=5)')
print(f'    The bilateral cut CONTAINED WITHIN the charge sector')
print(f'    produces the doublet pairing.')
print(f'')
print(f'  4D perm rep = 2 (doublet) + 1 + 1 (two singlets)')
print(f'  Doublet: the two charge types that transform into each other')
print(f'  Singlets: the two charge types that don\'t')
print(f'  Left-handed: doublet active (SU(2) acts)')
print(f'  Right-handed: doublet frozen (SU(2) trivial)')

S1: EXPLICIT Z_2 ≀ Z_2 = D_4 AND ITS 2-DIM IRREP
D_4 = Z_2 ≀ Z_2: 8 elements on 4 points

4D perm rep in symmetric/antisymmetric basis:
  Even parity (|+⟩) block = upper-left 2×2
  Odd parity (|-⟩) block = lower-right 2×2
  Max off-diagonal (block mixing): 0.00e+00

Representative 2×2 matrices (odd parity block):
  f=(0, 0), σ=0: [[1.0, 0.0], [0.0, 1.0]], det=1
  f=(0, 0), σ=1: [[0.0, 1.0], [1.0, 0.0]], det=-1
  f=(0, 1), σ=0: [[1.0, 0.0], [0.0, -1.0]], det=-1
  f=(0, 1), σ=1: [[0.0, -1.0], [1.0, 0.0]], det=1

Distinct 2×2 matrices: even=2, odd=8

Even block for (f=(1,0), σ=0):
  [[1. 0.]
 [0. 1.]]
Odd block for (f=(1,0), σ=0):
  [[-1.  0.]
 [ 0.  1.]]


THE 2-DIM IRREP (odd parity block):
       f   σ                matrix    det  order
  (0, 0)   0      [[1, 0], [0, 1]]     +1      1
  (0, 0)   1      [[0, 1], [1, 0]]     -1      2
  (0, 1)   0     [[1, 0], [0, -1]]     -1      2
  (0, 1)   1     [[0, -1], [1, 0]]     +1      4
  (1, 0)   0     [[-1, 0], [0, 1]]     -1      2
  (1, 0

## S2: Scorecard — The Complete Gauge Emergence Picture

### What NB144 establishes:

**Z₂ ≀ Z₂ = D₄** (dihedral group, order 8) has a **2-dimensional irreducible representation** — the SU(2) fundamental (the weak doublet).

The 4D permutation representation decomposes as **4 = 2 + 1 + 1**:
- **2-dim irrep** (odd parity block): the SU(2) doublet. Two charge types that transform into each other.
- **1 + 1** (even parity block): two singlets. Charge types that don't pair.

The det=+1 subgroup (4 elements) is Z₄ ⊂ SO(2) — the rotation subgroup. D₄ lifts to the binary dihedral group in SU(2).

### The unified wreath product mechanism:

| Gauge factor | Wreath product | Perm rep dim | Decomposition | Finite subgroup | Source in tower |
|---|---|---|---|---|---|
| **SU(3)** | Z₂ ≀ Z₃ (24) | 6D | **3** + 1+1+1 | A₄ ⊂ SU(3) | p₁=2 × p₂=3 |
| **SU(2)** | Z₂ ≀ Z₂ (8) | 4D | **2** + 1+1 | D₄ → SU(2) | p₁=2 × Z₂⊂Z₄ (from p₃=5) |
| **U(1)** | Z₄ (4) | — | 1+1+1+1 | Z₄ ⊂ U(1) | φ(p₃)=4 |

Both non-abelian gauge factors emerge from the **same mechanism**: the bilateral Z₂ (innermost orbit, p₁=2) forming a wreath product with a Z_n from a higher covering level. The innermost orbit is the common seed because it is contained within ALL outer orbits — its interaction with each outer level produces a different gauge factor.

### The pattern: n+1 = p_{k+1}

| Wreath | Z₂ ≀ Z_**n** | n+1 | Irrep dim | Gauge group SU(**n+1**) |
|---|---|---|---|---|
| SU(2) | Z₂ ≀ Z_**1** (=Z₂) | 2 | **2** | SU(**2**) |
| SU(3) | Z₂ ≀ Z_**2** (wait, Z₃?) | 3 | **3** | SU(**3**) |

Actually: Z₂ ≀ Z_q gives q-dim irreps. For SU(n): the n-dim fundamental comes from Z₂ ≀ Z_n's n-dim irrep — but we used Z₂ ≀ Z₃ for SU(3) (matching) and Z₂ ≀ Z₂ for SU(2) (matching). The pattern is: **Z₂ ≀ Z_n → SU(n)** through the n-dim irrep of the wreath product.

### What remains:

- The chirality dependence: WHY does SU(2) act only on left-handed fermions? The even/odd parity blocks of the decomposition provide the structure, but the selection of chirality (a₃ determines WHICH block is active) needs the containment structure from NB139.
- The complete 48-fermion bijection (Option 3, next notebook).